# GeoMarketing IDF — J3 : revenus et emploi

**Objectif :** enrichir le fichier démographique du J2 avec des indicateurs communaux de pouvoir d'achat et d'emploi.

Ce notebook :

1. télécharge la **Base du comparateur de territoires** de l'Insee ;
2. conserve uniquement les communes d'Île-de-France ;
3. repère automatiquement les indicateurs utiles à partir de leurs libellés ;
4. joint ces indicateurs au fichier du J2 avec `CODGEO` ;
5. contrôle notamment la commune nouvelle de Saint-Denis ;
6. exporte une table socio-économique propre.

Source officielle : [Base du comparateur de territoires — Insee](https://www.insee.fr/fr/statistiques/2521169), parue le 7 août 2026. La base utilise la géographie communale au 1er janvier 2026.

## 1. Préparer Python

Cette cellule corrige également l'erreur déjà rencontrée avec `OMP_NUM_THREADS='A'`. Si `pandas` a déjà échoué dans ce noyau, utilise **Kernel → Restart Kernel**, puis relance toutes les cellules.

In [1]:
import os

for variable in ("OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS", "MKL_NUM_THREADS"):
    valeur = os.environ.get(variable)
    if valeur is not None:
        try:
            int(valeur)
        except ValueError:
            print(f"Variable invalide supprimée : {variable}={valeur!r}")
            os.environ.pop(variable, None)

from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile
import io
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
print("Environnement Python prêt.")

Variable invalide supprimée : OMP_NUM_THREADS='A'
Environnement Python prêt.


## 2. Retrouver le dossier du projet

Le notebook cherche automatiquement `GeoMarketing_IDF` dans les emplacements OneDrive les plus courants. Modifie seulement `DOSSIER_PROJET_MANUEL` si la recherche échoue.

In [2]:
DOSSIER_PROJET_MANUEL = None
# Exemple : Path(r"C:\Users\VotreNom\OneDrive\GeoMarketing_IDF")

def trouver_projet():
    if DOSSIER_PROJET_MANUEL is not None:
        return Path(DOSSIER_PROJET_MANUEL)

    candidats = []
    for variable in ("OneDrive", "OneDriveConsumer", "OneDriveCommercial"):
        racine = os.environ.get(variable)
        if racine:
            candidats.extend([
                Path(racine) / "GeoMarketing_IDF",
                Path(racine) / "Documents" / "GeoMarketing_IDF",
            ])

    candidats.extend([
        Path.home() / "OneDrive" / "GeoMarketing_IDF",
        Path.home() / "Documents" / "GeoMarketing_IDF",
        Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd(),
    ])

    for candidat in candidats:
        if (candidat / "data").exists():
            return candidat.resolve()

    raise FileNotFoundError(
        "Dossier GeoMarketing_IDF introuvable. Renseigne DOSSIER_PROJET_MANUEL."
    )

PROJET = trouver_projet()
RAW = PROJET / "data" / "raw" / "insee"
PROCESSED = PROJET / "data" / "processed" / "insee"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

FICHIER_J2 = PROCESSED / "indicateurs_demographiques_idf_2022.csv"
FICHIER_SORTIE = PROCESSED / "profil_socioeconomique_idf.csv"

print("Projet :", PROJET)
print("Entrée J2 :", FICHIER_J2)
print("Sortie J3 :", FICHIER_SORTIE)
assert FICHIER_J2.exists(), f"Fichier J2 absent : {FICHIER_J2}"


Projet : C:\Users\almou\OneDrive\GeoMarketing_IDF
Entrée J2 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\indicateurs_demographiques_idf_2022.csv
Sortie J3 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\profil_socioeconomique_idf.csv


## 3. Télécharger et ouvrir la base Insee

Le téléchargement n'est effectué qu'une seule fois. En cas de blocage réseau, télécharge manuellement le fichier CSV depuis la page Insee et place l'archive sous le nom `comparateur_csv.zip` dans `data/raw/insee/`.

In [3]:
URL_COMPARATEUR = "https://www.insee.fr/fr/statistiques/fichier/2521169/comparateur_csv.zip"
ARCHIVE = RAW / "comparateur_csv.zip"

if not ARCHIVE.exists():
    print("Téléchargement de la base Insee (environ 17 Mo)...")
    requete = Request(URL_COMPARATEUR, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(requete, timeout=180) as reponse, open(ARCHIVE, "wb") as sortie:
        while True:
            bloc = reponse.read(1024 * 1024)
            if not bloc:
                break
            sortie.write(bloc)
    print("Téléchargement terminé.")
else:
    print("Archive déjà présente :", ARCHIVE)

assert ARCHIVE.stat().st_size > 1_000_000, "L'archive semble incomplète. Supprime-la puis relance."

def detecter_fichiers(archive):
    requis_donnees = {"GEO_OBJECT", "GEO", "TIME_PERIOD", "DS", "TAB_MEASURE", "OBS_VALUE"}
    requis_metadata = {"COD_VAR", "COD_MOD", "LIB_MOD"}
    trouve_donnees = None
    trouve_metadata = None
    with ZipFile(archive) as z:
        csvs = sorted(
            (i for i in z.infolist() if i.filename.lower().endswith(".csv")),
            key=lambda i: i.file_size,
            reverse=True,
        )
        for info in csvs:
            brut = z.read(info.filename)[:10000]
            texte = brut.decode("utf-8-sig", errors="replace")
            sep = ";" if texte.splitlines()[0].count(";") > texte.splitlines()[0].count(",") else ","
            colonnes = set(pd.read_csv(io.StringIO(texte), sep=sep, nrows=0).columns.str.strip())
            if requis_donnees.issubset(colonnes):
                trouve_donnees = (info.filename, sep)
            elif requis_metadata.issubset(colonnes):
                trouve_metadata = (info.filename, sep)
    if trouve_donnees is None:
        raise ValueError("CSV principal non reconnu dans l'archive Insee.")
    if trouve_metadata is None:
        raise ValueError("CSV des métadonnées non reconnu dans l'archive Insee.")
    return trouve_donnees, trouve_metadata

(nom_csv, separateur), (nom_metadata, sep_metadata) = detecter_fichiers(ARCHIVE)
print("CSV de données détecté :", nom_csv)
print("CSV de métadonnées détecté :", nom_metadata)

with ZipFile(ARCHIVE) as z:
    with z.open(nom_csv) as fichier:
        morceaux = []
        for bloc in pd.read_csv(
            fichier, sep=separateur, dtype=str, encoding="utf-8-sig",
            chunksize=250_000, low_memory=False
        ):
            # On réduit immédiatement la base de 193 Mo aux seules communes IDF.
            masque = bloc["GEO_OBJECT"].eq("COM") & bloc["GEO"].str[:2].isin(
                {"75", "77", "78", "91", "92", "93", "94", "95"}
            )
            morceaux.append(bloc.loc[masque].copy())
        comparateur = pd.concat(morceaux, ignore_index=True)

    with z.open(nom_metadata) as fichier:
        metadata = pd.read_csv(
            fichier, sep=sep_metadata, dtype=str, encoding="utf-8-sig", low_memory=False
        )

comparateur.columns = comparateur.columns.str.strip()
metadata.columns = metadata.columns.str.strip()

# Ajouter les libellés des mesures, stockés séparément depuis la version 2026.
lib_mesures = (
    metadata.loc[metadata["COD_VAR"].eq("TAB_MEASURE"), ["COD_MOD", "LIB_MOD"]]
    .drop_duplicates("COD_MOD")
    .rename(columns={"COD_MOD": "TAB_MEASURE", "LIB_MOD": "TAB_MEASURE_LABEL"})
)
comparateur = comparateur.merge(lib_mesures, on="TAB_MEASURE", how="left", validate="many_to_one")

# Ajouter les noms des communes depuis les métadonnées.
masque_geo = metadata["COD_VAR"].eq("GEO")
if "GEO_OBJECT" in metadata.columns:
    masque_geo &= metadata["GEO_OBJECT"].eq("COM")
lib_communes = (
    metadata.loc[masque_geo, ["COD_MOD", "LIB_MOD"]]
    .drop_duplicates("COD_MOD")
    .rename(columns={"COD_MOD": "GEO", "LIB_MOD": "GEO_LABEL"})
)
comparateur = comparateur.merge(lib_communes, on="GEO", how="left", validate="many_to_one")
comparateur["GEO_LABEL"] = comparateur["GEO_LABEL"].fillna(comparateur["GEO"])

print(f"{len(comparateur):,} lignes communales IDF chargées.")
display(comparateur.head())

Archive déjà présente : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\comparateur_csv.zip
CSV de données détecté : comparateur.csv
CSV de métadonnées détecté : comparateur_metadata.csv
88,620 lignes communales IDF chargées.


,GEO_REF,GEO_OBJECT,GEO,TIME_PERIOD,DS,TAB_MEASURE,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,UNIT_MULT,CONF_STATUS,TAB_MEASURE_LABEL,GEO_LABEL
0,2026,COM,75056,2025,DS_ETAT_CIVIL_NAIS_COMMUNES,LVB,21189,A,NaN,NaN,NaN,Nombre de naissances vivantes,Paris
1,2026,COM,75056,2025,DS_ETAT_CIVIL_DECES_COMMUNES,DTH,13878,A,NaN,NaN,NaN,Nombre de décès,Paris
2,2025,COM,75056,2024,DS_FLORES_A5,UNIT_LOC,188002,A,NaN,NaN,NaN,Établissements,Paris
3,2025,COM,75056,2024,DS_FLORES_A5,UNIT_LOC_ACTIVITY_OQ,12676,A,NaN,NaN,NaN,"Établissements – Administration publique, ense...",Paris
4,2025,COM,75056,2024,DS_FLORES_A5,UNIT_LOC_NUMBER_EMPL_E1T4,109078,A,NaN,NaN,NaN,Établissements – 1 à 4 salariés,Paris


## 4. Filtrer automatiquement les communes franciliennes

Les codes sont conservés en texte : on ne transforme jamais `CODGEO` en entier.

In [4]:
colonnes_requises = {
    "GEO_OBJECT", "GEO", "GEO_LABEL", "TIME_PERIOD",
    "TAB_MEASURE", "TAB_MEASURE_LABEL", "OBS_VALUE"
}
manquantes = colonnes_requises - set(comparateur.columns)
assert not manquantes, f"Colonnes manquantes : {sorted(manquantes)}"

idf_departements = {"75", "77", "78", "91", "92", "93", "94", "95"}

communes = comparateur.loc[comparateur["GEO_OBJECT"].eq("COM")].copy()
communes["CODGEO"] = communes["GEO"].str.extract(r"(\d{5})$")[0]
communes = communes.loc[communes["CODGEO"].str[:2].isin(idf_departements)].copy()

communes["OBS_VALUE_NUM"] = pd.to_numeric(
    communes["OBS_VALUE"].str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce",
)

print("Communes franciliennes distinctes :", communes["CODGEO"].nunique())
print("Périodes présentes :", sorted(communes["TIME_PERIOD"].dropna().unique()))
assert communes["CODGEO"].nunique() > 1200, "Le filtre IDF a produit trop peu de communes."
display(communes[["CODGEO", "GEO_LABEL"]].drop_duplicates().head())

Communes franciliennes distinctes : 1266
Périodes présentes : ['1968', '1975', '1982', '1990', '1999', '2007', '2012', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


,CODGEO,GEO_LABEL
0,75056,Paris
70,77001,Achères-la-Forêt
140,77002,Amillis
210,77003,Amponville
280,77004,Andrezel


## 5. Repérer les indicateurs utiles

La base 2026 est en format long. On utilise les codes officiels des mesures, puis on affiche leur libellé pour contrôler leur signification.

In [5]:
def normaliser(texte):
    texte = "" if pd.isna(texte) else str(texte)
    texte = re.sub(r"[’'`´]", " ", texte)
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode("ascii")
    texte = re.sub(r"[^a-z0-9]+", " ", texte.lower())
    return re.sub(r"\s+", " ", texte).strip()

mesures = communes[["TAB_MEASURE", "TAB_MEASURE_LABEL"]].drop_duplicates().copy()

selection = {
    "REVENU_MEDIAN_EUROS": "MED_SL",
    "TAUX_PAUVRETE_PCT": "PR_MD60",
    "EMPLOIS_SALARIES_LIEU_TRAVAIL": "NBEMP_EMPFORM_2",
    "NOMBRE_ETABLISSEMENTS": "UNIT_LOC",
}

codes_disponibles = set(mesures["TAB_MEASURE"])
codes_absents = set(selection.values()) - codes_disponibles
assert not codes_absents, f"Mesures Insee absentes : {sorted(codes_absents)}"

for nom, code in selection.items():
    libelle = mesures.loc[mesures["TAB_MEASURE"].eq(code), "TAB_MEASURE_LABEL"].iloc[0]
    print(f"✓ {nom} <- {libelle} [{code}]")

✓ REVENU_MEDIAN_EUROS <- Niveau de vie médian (en euros) [MED_SL]
✓ TAUX_PAUVRETE_PCT <- Taux de pauvreté (en %) au seuil de 60 % de la médiane du niveau de vie [PR_MD60]
✓ EMPLOIS_SALARIES_LIEU_TRAVAIL <- Nombre d’emplois – Salariés [NBEMP_EMPFORM_2]
✓ NOMBRE_ETABLISSEMENTS <- Établissements [UNIT_LOC]


## 6. Mettre les indicateurs en colonnes

Pour chaque indicateur, on conserve automatiquement la période disponible la plus récente. La période est également enregistrée pour éviter de mélanger silencieusement les millésimes.

In [6]:
resultats = None
periodes_utilisees = {}

for nouveau_nom, code_mesure in selection.items():
    extrait = communes.loc[communes["TAB_MEASURE"].eq(code_mesure)].copy()
    periodes = extrait["TIME_PERIOD"].dropna().unique()
    annee = sorted(periodes)[-1]
    periodes_utilisees[nouveau_nom] = annee
    extrait = extrait.loc[extrait["TIME_PERIOD"].eq(annee)]

    doublons = extrait.duplicated("CODGEO", keep=False)
    if doublons.any():
        print(f"Attention : doublons agrégés pour {nouveau_nom} ({doublons.sum()} lignes).")

    serie = (
        extrait.groupby("CODGEO", as_index=False)
        .agg(GEO_LABEL=("GEO_LABEL", "first"), **{nouveau_nom: ("OBS_VALUE_NUM", "first")})
    )
    if resultats is None:
        resultats = serie
    else:
        resultats = resultats.merge(
            serie.drop(columns="GEO_LABEL"), on="CODGEO", how="outer", validate="one_to_one"
        )

print("Périodes utilisées :")
for indicateur, periode in periodes_utilisees.items():
    print(f"- {indicateur} : {periode}")

assert resultats["CODGEO"].is_unique
display(resultats.head())

Périodes utilisées :
- REVENU_MEDIAN_EUROS : 2023
- TAUX_PAUVRETE_PCT : 2023
- EMPLOIS_SALARIES_LIEU_TRAVAIL : 2023
- NOMBRE_ETABLISSEMENTS : 2024


,CODGEO,GEO_LABEL,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS
0,75056,Paris,33650.0,16.8,1.551134e+06,188002.0
1,77001,Achères-la-Forêt,34840.0,NaN,1.303068e+02,35.0
2,77002,Amillis,29130.0,NaN,1.906978e+02,39.0
3,77003,Amponville,30310.0,NaN,1.995946e+01,12.0
4,77004,Andrezel,31970.0,NaN,2.316751e+01,13.0


## 7. Joindre avec le fichier démographique du J2

In [12]:
def lire_csv_robuste(chemin):
    for encodage in ("utf-8-sig", "utf-8", "cp1252"):
        for sep in (";", ","):
            try:
                df = pd.read_csv(chemin, sep=sep, dtype=str, encoding=encodage)
                if len(df.columns) > 1:
                    return df
            except UnicodeDecodeError:
                continue
    raise ValueError(f"Impossible de lire {chemin}")

demographie = lire_csv_robuste(FICHIER_J2)
demographie.columns = demographie.columns.str.strip()
assert "CODGEO" in demographie.columns, "CODGEO absent du fichier J2."
demographie["CODGEO"] = demographie["CODGEO"].astype("string").str.strip().str.zfill(5)
assert demographie["CODGEO"].is_unique, "CODGEO n'est pas unique dans le fichier J2."

profil = demographie.merge(
    resultats.drop(columns="GEO_LABEL"),
    on="CODGEO",
    how="left",
    validate="one_to_one",
    indicator=True,
)

taux_jointure = profil["_merge"].eq("both").mean()
print(f"Taux de jointure : {taux_jointure:.1%}")
display(profil.loc[profil["_merge"].ne("both"), ["CODGEO", "_merge"]].head(20))
assert taux_jointure > 0.98, "Moins de 98 % des communes ont été jointes : vérifier les millésimes géographiques."
profil = profil.drop(columns="_merge")


Taux de jointure : 100.0%


,CODGEO,_merge


## 8. Créer quelques ratios simples

Ces ratios servent à explorer les communes. Ils ne constituent pas encore le score final d'implantation.

In [9]:
def colonne_population(df):
    candidats = ["POP_2022", "P22_POP", "POPULATION_2022"]
    for col in candidats:
        if col in df.columns:
            return col
    return None

col_pop = colonne_population(profil)
colonnes_numeriques = list(selection.keys())
if col_pop:
    colonnes_numeriques.append(col_pop)

for col in colonnes_numeriques:
    if col in profil.columns:
        profil[col] = pd.to_numeric(
            profil[col].astype("string").str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
            errors="coerce",
        )

if col_pop and "EMPLOIS_SALARIES_LIEU_TRAVAIL" in profil.columns:
    profil["EMPLOIS_SALARIES_POUR_100_HAB"] = 100 * profil["EMPLOIS_SALARIES_LIEU_TRAVAIL"] / profil[col_pop]

if col_pop and "NOMBRE_ETABLISSEMENTS" in profil.columns:
    profil["ETABLISSEMENTS_POUR_1000_HAB"] = 1000 * profil["NOMBRE_ETABLISSEMENTS"] / profil[col_pop]

print("Dimensions finales :", profil.shape)
display(profil.head())

Dimensions finales : (1266, 34)


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508206,513386.75602884,457511.658563684,387099.215350194,302819.494109117,154040.332161959,24566.9822779995,787667.317537046,970898.4145925241,481426.8085490755,178607.31443995849,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.5,-0.59,33650.0,16.8,1551133.75022,188002.0,73.38459,88.944294
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135094788161,159.522993092748,189.869837113436,347.331906368833,204.113784168138,89.442063221765,10.5843212469189,341.658087880909,349.39283020618404,304.1401686368219,100.0263844686839,15.4,13.48,16.05,28.88,29.53,25.71,8.46,44.0,3.86,0.63,34840.0,<NA>,130.30679,35.0,11.014944,29.585799
2,77002,Amillis,77,11,781.0,819.0,821.0,127.936213516012,122.351987081097,146.913593456989,151.60336129515,188.399316742963,59.3206016148823,24.4749262929075,250.28820059710898,269.265580538086,272.1948446507528,83.7955279077898,15.58,14.9,17.89,30.49,32.8,33.15,10.21,2.0,0.24,0.04,29130.0,<NA>,190.69785,39.0,23.227509,47.503045
3,77003,Amponville,77,11,388.0,351.0,351.0,63.0,38.0,72.0,88.0,62.0,28.0,0.0,101.0,110.0,90.0,28.0,17.95,10.83,20.51,28.77,31.34,25.64,7.98,0.0,0.0,0.0,30310.0,<NA>,19.95946,12.0,5.686456,34.188034
4,77004,Andrezel,77,11,299.0,285.0,324.0,77.0,33.0,78.0,53.0,67.0,15.0,1.0,110.0,111.0,83.0,16.0,23.77,10.19,24.07,33.95,34.26,25.62,4.94,39.0,13.68,2.16,31970.0,<NA>,23.16751,13.0,7.150466,40.123457


## 9. Contrôler Saint-Denis et quelques communes tests

Après la fusion, `93066` doit représenter la commune nouvelle de Saint-Denis et `93059` ne doit plus apparaître comme commune indépendante.

In [10]:
assert "93066" in set(profil["CODGEO"]), "Saint-Denis (93066) est absent."
assert "93059" not in set(profil["CODGEO"]), "Pierrefitte-sur-Seine (93059) apparaît encore séparément."

communes_test = ["93066", "95680", "95268", "95018"]  # Saint-Denis, Villiers-le-Bel, Garges, Argenteuil
colonnes_affichage = [
    c for c in [
        "CODGEO", "LIBGEO", "NOM_COM", "REVENU_MEDIAN_EUROS", "TAUX_PAUVRETE_PCT",
        "EMPLOIS_SALARIES_LIEU_TRAVAIL", "EMPLOIS_SALARIES_POUR_100_HAB",
        "NOMBRE_ETABLISSEMENTS", "ETABLISSEMENTS_POUR_1000_HAB"
    ] if c in profil.columns
]
display(profil.loc[profil["CODGEO"].isin(communes_test), colonnes_affichage])
print("Contrôle des communes fusionnées réussi.")

,CODGEO,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,EMPLOIS_SALARIES_POUR_100_HAB,NOMBRE_ETABLISSEMENTS,ETABLISSEMENTS_POUR_1000_HAB
1027,93066,18130.0,38.6,94458.63073,63.434648,4912.0,32.987032
1088,95018,21300.0,29.8,26871.99225,25.082365,3126.0,29.17814
1157,95268,17320.0,40.8,7383.02074,17.417714,937.0,22.105313
1263,95680,17750.0,39.4,4662.38655,15.946325,638.0,21.820918


Contrôle des communes fusionnées réussi.


## 10. Contrôles qualité et export

In [11]:
assert profil["CODGEO"].is_unique
assert profil["CODGEO"].str.fullmatch(r"\d{5}").all()

for col in ["TAUX_PAUVRETE_PCT", "TAUX_ACTIVITE_15_64_PCT", "TAUX_CHOMAGE_15_64_PCT"]:
    if col in profil.columns:
        valeurs = profil[col].dropna()
        assert valeurs.between(0, 100).all(), f"Valeurs hors limites dans {col}."

if "REVENU_MEDIAN_EUROS" in profil.columns:
    assert (profil["REVENU_MEDIAN_EUROS"].dropna() > 0).all()

profil.to_csv(FICHIER_SORTIE, index=False, encoding="utf-8-sig")
print("Fichier créé :", FICHIER_SORTIE)
print("Nombre de communes :", len(profil))
print("J3 terminé ✅")

Fichier créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\profil_socioeconomique_idf.csv
Nombre de communes : 1266
J3 terminé ✅


## Résultat du J3

Le fichier suivant est maintenant disponible dans OneDrive :

`data/processed/insee/profil_socioeconomique_idf.csv`

Il contient les indicateurs démographiques du J2 enrichis par les revenus et l'emploi. Les valeurs manquantes de certains indicateurs ne sont pas forcément des erreurs : l'Insee applique le secret statistique aux petites communes.

Enregistre ensuite ce notebook dans le dépôt GitHub sous :

`notebooks/03_revenus_emploi_idf.ipynb`

Résumé du commit : `Ajout des indicateurs de revenus et d'emploi IDF`